# 5 Laufzeitmodell

# 5.2 Kompilieren und Ausführen

## Python als Mischmodell (8)

```text
 Quelltext
   |
   v
 AST (Abstract Syntax Tree)
   |
   v
 Bytecode
   |
   v
 Python Virtual Machine
```

In [ ]:
import ast

source = "x = 1 + 2 * 3"
tree = ast.parse(source)
print(ast.dump(tree, indent=2))

In [ ]:
import dis

code = compile(source, "<demo>", "exec")
dis.dis(code)

## Laufzeitobjekte in Python (9)

In [ ]:
def greet(name):
    return f"Hallo {name}"

alias = greet
greet.category = "demo"

print(alias("Ada"))
print("function object attribute:", greet.category)

In [ ]:
class Counter:
    pass

Counter.version = "1.0"

print("class attribute:", Counter.version)

In [ ]:
car_counter = Counter()
car_counter.color = "red"

print("instance attribute:", car_counter.color)

In [ ]:
bike_counter = Counter()
bike_counter.version = "2.0"

print("class attribute:", Counter.version)
print("instance attribute:", bike_counter.version)

In [ ]:
def make_greet(greeting):
    def greet(name):
        return f"{greeting}, {name}"
    return greet

greet_english = make_greet("Hello")
greet_german = make_greet("Guten Tag")

print(greet_english("Alice"))
print(greet_german("Sophie"))

# 5.3 Speicherbereiche

## Call Stack (11)

In [ ]:
import inspect

def print_stack():
    for i, frame in enumerate(inspect.stack()):
        print(f"{i}: {frame.function}() in {frame.filename}:{frame.lineno}, {frame.frame.f_locals}")
        if i >= 5:
            print("...")
            break

def factorial(n):
    if n == 1:
        print_stack()
        return 1
    return n * factorial(n - 1)

factorial(4)

## Stack und Heap in Python (13)

Beispiel:

```text
      Stack                           Heap

Frame build_numbers():
  numbers (Name) -----------------> [1, 2, 3] (Objekt)
Frame ipykernel():
  ...
```


In [ ]:
def build_numbers():
    numbers = [1, 2, 3]
    print("inside function:", id(numbers), numbers)
    return numbers

result = build_numbers()

print("outside function:", id(result), result)

# 5.4 Objekterzeugung und Lebensdauer

## Lebensdauer von Objekten (17)

In [ ]:
import gc
import weakref

class Box:
    def __init__(self, value):
        self.value = value

box = Box([1, 2, 3])
alias = box
ref = weakref.ref(box)

del box
print("after del box, object alive:", ref() is not None)

del alias
gc.collect()
print("after del alias, object alive:", ref() is not None)

# 5.5 Garbage Collection

## Zyklische Garbage Collection (22)

In [ ]:
import gc
import weakref

class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

gc.disable()

a = Node("A")
b = Node("B")
a.next = b
b.next = a

wa = weakref.ref(a)
wb = weakref.ref(b)

del a
del b
print("before gc.collect():", wa() is not None, wb() is not None)

gc.collect()
print("after gc.collect():", wa() is not None, wb() is not None)

gc.enable()

## Speicher und andere Ressourcen (24)

Speicher und andere Ressourcen haben oft unterschiedliche Anforderungen.

| Thema | Typische Freigabe |
| --- | --- |
| Speicher | durch Garbage Collection |
| Datei-Handle | möglichst sofort nach Nutzung |
| Socket / DB-Verbindung | möglichst sofort nach Nutzung |

In [ ]:
from pathlib import Path

tmp_path = Path("runtime_model_demo.txt")
with open(tmp_path, "w", encoding="utf-8") as handle:
    handle.write("temporary data\n")
    print("inside with, handle.closed =", handle.closed)

print("outside with, handle.closed =", handle.closed)
tmp_path.unlink()

# 5.6 Explizite Speicherverwaltung

## Vergleich der Ansätze (26)

| Aspekt | Python | C++ |
| --- | --- | --- |
| Speicherfreigabe | automatisch durch Laufzeitsystem | expliziter und direkter kontrollierbar |
| typische Gefahr | unbeabsichtigt festgehaltene Referenzen | Leaks, Dangling Pointers, Double Free |
| praktische Konsequenz | Referenzdisziplin und Ressourcenmanagement bleiben wichtig | mehr Kontrolle, aber auch mehr Verantwortung |

# Übungen zu Kapitel 5

## Aufgabe zu 5.2 - AST und Bytecode vergleichen

1. Lege einen Quelltext `source = "total = (4 + 5) * 2"` an.
2. Parse ihn mit `ast.parse(...)`.
3. Kompiliere ihn mit `compile(..., "exec")`.
4. Gib zuerst den AST strukturiert und danach den Bytecode aus.

## Aufgabe zu 5.3 - Referenzen beobachten

1. Schreibe eine Funktion `append_marker(values, marker)`.
2. Die Funktion soll die Liste verändern, ihre `id(...)` ausgeben und die Liste zurückgeben.
3. Rufe die Funktion mit einer Liste aus dem aufrufenden Kontext auf.
4. Prüfe danach, ob Rückgabewert und ursprüngliche Liste dasselbe Objekt sind.

## Aufgabe zu 5.4 - Lebensdauer mit `weakref`

1. Definiere eine Klasse `Payload`.
2. Erzeuge ein Objekt, einen Alias und eine `weakref.ref(...)` darauf.
3. Lösche zuerst nur einen Namen und prüfe, ob das Objekt noch lebt.
4. Lösche danach auch den Alias, rufe `gc.collect()` auf und prüfe erneut.

## Aufgabe zu 5.5 - Referenzzyklus erzeugen

1. Definiere eine Klasse `Node` mit dem Attribut `next`.
2. Erzeuge zwei Knoten, die sich gegenseitig referenzieren.
3. Lege `weakref`-Referenzen an.
4. Lösche die starken Referenzen und zeige mit `gc.collect()`, dass der Zyklus beseitigt werden kann.

## Aufgabe zu 5.6 - Ressourcen mit Context Manager

1. Implementiere eine Klasse `ManagedResource`.
2. Die Klasse soll in `__enter__` den Zustand `opened = True` setzen und sich selbst zurückgeben.
3. In `__exit__` soll `opened = False` gesetzt werden.
4. Verwende die Klasse in einem `with`-Block und gib den Zustand innerhalb und außerhalb des Blocks aus.